# Sesión 12 — Redes Feedforward y Retropropagación
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo IV · Fundamentos de Deep Learning**

## Marco del módulo

Los Módulos I–III te dieron las bases estadísticas y probabilísticas.
El Módulo IV construye el **motor del deep learning** — partiendo de una sola neurona,
derivando la retropropagación a mano, y añadiendo sistemáticamente la maquinaria
(normalización, regularización, optimizadores) necesaria para entrenar redes biomédicas reales.

La progresión a lo largo de cuatro sesiones es deliberada:
- **Sesión 12** — construir una red desde cero, derivar retropropagación, verificar gradientes
- **Sesión 13** — redes convolucionales para señales y series temporales biomédicas
- **Sesión 14** — redes recurrentes para modelado temporal
- **Sesión 15** — regularización, prácticas de entrenamiento e interpretabilidad (Grad-CAM)

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Comparar funciones de activación y diagnosticar el problema del gradiente que se desvanece (vanishing gradient).
2. Implementar el forward pass y la retropropagación de un MLP completamente desde cero en NumPy.
3. Verificar una implementación de backprop mediante gradient checking numérico.
4. Implementar el optimizador Adam desde cero y compararlo con SGD.
5. Replicar la red en PyTorch y verificar la consistencia con autograd.
6. Demostrar el teorema de aproximación universal variando el ancho de la capa oculta.

## Conjunto de datos principal

**Clasificación EEG banda alfa** — reposo vs imaginería motora.
10 características: potencia en 5 bandas de frecuencia × 2 hemisferios.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*. Cap. 6 (Redes Feedforward). MIT Press. https://www.deeplearningbook.org/ |
| ★★★ | Rumelhart, D.E., Hinton, G.E. & Williams, R.J. (1986). Learning representations by back-propagating errors. *Nature*, 323, 533–536. |
| ★★☆ | Kingma, D.P. & Ba, J. (2015). Adam: A method for stochastic optimization. *ICLR 2015*. https://arxiv.org/abs/1412.6980 |
| ★★☆ | Hornik, K., Stinchcombe, M. & White, H. (1989). Multilayer feedforward networks are universal approximators. *Neural Networks*, 2(5), 359–366. |
| ★☆☆ | Karpathy, A. micrograd. https://github.com/karpathy/micrograd |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(0)
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})

# Verificar disponibilidad de PyTorch
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    HAS_TORCH = True
    print('PyTorch disponible.')
except ImportError:
    HAS_TORCH = False
    print('PyTorch no disponible — instalar con: pip install torch --break-system-packages')

## Parte 1 — Funciones de activación y el problema del gradiente que se desvanece

La función de activación determina qué puede representar una neurona, y de forma crítica,
si los gradientes pueden fluir a través de muchas capas.

In [ ]:
x = np.linspace(-5, 5, 500)

activaciones = {
    'Sigmoide  σ(x)':  (lambda z: 1/(1+np.exp(-z)),
                         lambda z: (1/(1+np.exp(-z))) * (1 - 1/(1+np.exp(-z)))),
    'Tanh':             (np.tanh,
                         lambda z: 1 - np.tanh(z)**2),
    'ReLU':             (lambda z: np.maximum(0, z),
                         lambda z: (z > 0).astype(float)),
    'Leaky ReLU':       (lambda z: np.where(z > 0, z, 0.01*z),
                         lambda z: np.where(z > 0, 1.0, 0.01)),
    'GELU':             (lambda z: z * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi)*(z+0.044715*z**3))),
                         None),   # derivada compleja — se calcula numéricamente
}

fig, axes = plt.subplots(2, len(activaciones), figsize=(18, 7))

for col, (nombre, (fn, dfn)) in enumerate(activaciones.items()):
    y = fn(x)
    axes[0, col].plot(x, y, lw=2.5, color='steelblue')
    axes[0, col].axhline(0, color='gray', lw=0.8, ls='--')
    axes[0, col].axvline(0, color='gray', lw=0.8, ls='--')
    axes[0, col].set(title=nombre, ylim=(-1.6, 1.6), xlabel='x')
    if col == 0: axes[0, col].set_ylabel('f(x)')

    if dfn is not None:
        dy = dfn(x)
    else:
        dy = np.gradient(fn(x), x)

    axes[1, col].plot(x, dy, lw=2.5, color='tomato')
    axes[1, col].axhline(0, color='gray', lw=0.8, ls='--')
    # Resaltar zonas de saturación
    if 'Sigmoide' in nombre or 'Tanh' in nombre:
        axes[1, col].fill_between(x, dy,
                                    where=(np.abs(x) > 3),
                                    alpha=0.3, color='tomato',
                                    label='Saturación\n(gradiente ≈ 0)')
        axes[1, col].legend(fontsize=7)
    axes[1, col].set(xlabel='x', ylim=(-0.1, 1.1))
    if col == 0: axes[1, col].set_ylabel("f'(x)")

fig.suptitle('Funciones de activación y sus derivadas\n'
             'Rojo = gradiente ("¿cuánta señal fluye de regreso a través de esta neurona?")', y=1.01)
plt.tight_layout()
plt.show()

# Demostración del gradiente que se desvanece con sigmoide en 20 capas
print('\nDemostración del gradiente que se desvanece — cadena de sigmoides:')
print('∂L/∂x₁ ≈ (producto de 20 derivadas de sigmoide evaluadas en z=0)')
sig_deriv_en_0 = 0.25   # σ'(0) = 0.25
for n_capas in [1, 5, 10, 20]:
    grad = sig_deriv_en_0 ** n_capas
    print(f'  Capas = {n_capas:3d}:  gradiente ≈ {grad:.2e}')

## Parte 2 — Forward pass desde cero

Un MLP de $L$ capas calcula:
$$\mathbf{z}^{(l)} = \mathbf{W}^{(l)}\mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}, \qquad \mathbf{a}^{(l)} = g^{(l)}(\mathbf{z}^{(l)})$$
con $\mathbf{a}^{(0)} = \mathbf{x}$.

In [ ]:
# ── MLP mínimo en NumPy — clasificación binaria ───────────────────────────────

def sigmoide(z):    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def relu(z):        return np.maximum(0, z)
def relu_grad(z):   return (z > 0).astype(float)

class MLP_NumPy:
    """
    Perceptrón multicapa con capas ocultas ReLU y salida sigmoide.
    Capas: [d_entrada, *tamaños_ocultos, 1]
    """
    def __init__(self, tamanos_capa, lr=0.01, seed=0):
        rng_w = np.random.default_rng(seed)
        self.lr = lr
        self.weights = []
        self.biases  = []
        for i in range(len(tamanos_capa)-1):
            fan_in  = tamanos_capa[i]
            fan_out = tamanos_capa[i+1]
            # Inicialización He (para ReLU)
            scale = np.sqrt(2.0 / fan_in)
            self.weights.append(rng_w.normal(0, scale, (fan_in, fan_out)))
            self.biases.append(np.zeros(fan_out))
        self.L = len(self.weights)

    def forward(self, X):
        """Forward pass. Almacena activaciones pre/post para backprop."""
        self._cache = {'a': [X], 'z': []}
        a = X
        for l in range(self.L):
            z = a @ self.weights[l] + self.biases[l]
            self._cache['z'].append(z)
            # ReLU en capas ocultas, sigmoide en la salida
            a = sigmoide(z) if l == self.L-1 else relu(z)
            self._cache['a'].append(a)
        return a.squeeze()   # (N,)

    def backward(self, y):
        """Retropropagación — calcula gradientes vía regla de la cadena."""
        N = len(y)
        p = self._cache['a'][-1].squeeze()

        # Capa de salida: ∂L/∂z_out = (p - y) / N  [entropía cruzada + sigmoide combinados]
        delta = (p - y)[:, None] / N

        grads_W, grads_b = [None]*self.L, [None]*self.L

        for l in reversed(range(self.L)):
            a_prev = self._cache['a'][l]
            grads_W[l] = a_prev.T @ delta
            grads_b[l] = delta.sum(axis=0)

            if l > 0:   # propagar a la capa oculta l
                delta = (delta @ self.weights[l].T) * relu_grad(self._cache['z'][l-1])

        return grads_W, grads_b

    def update(self, grads_W, grads_b):
        """Paso de SGD estándar."""
        for l in range(self.L):
            self.weights[l] -= self.lr * grads_W[l]
            self.biases[l]  -= self.lr * grads_b[l]

    def loss(self, y, p_hat, eps=1e-12):
        p = np.clip(p_hat, eps, 1-eps)
        return -np.mean(y * np.log(p) + (1-y) * np.log(1-p))

    def fit(self, X, y, n_epochs=300, verbose=True):
        history = []
        for ep in range(n_epochs):
            p_hat = self.forward(X)
            loss  = self.loss(y, p_hat)
            gW, gb = self.backward(y)
            self.update(gW, gb)
            history.append(loss)
            if verbose and (ep+1) % 50 == 0:
                print(f'  Época {ep+1:4d}  pérdida={loss:.5f}')
        return history

    def predict_proba(self, X):
        return self.forward(X)

print('Clase MLP_NumPy definida.')

In [ ]:
# ── Dataset: clasificación EEG banda alfa (reposo vs imaginería motora) ──────
# 10 características: potencia de 5 bandas de frecuencia × 2 hemisferios
n_reposo = 300; n_im = 300
feat_dim = 10

X_reposo_nn = rng.multivariate_normal(np.zeros(feat_dim),
                                       np.eye(feat_dim), n_reposo)
X_im_nn     = rng.multivariate_normal(
    np.array([0, 0, -1.5, 1.2, 0, 0, 0, -1.5, 1.0, 0]),   # cambios alfa-beta contralaterales
    np.eye(feat_dim) * 1.2, n_im)

X_nn = np.vstack([X_reposo_nn, X_im_nn])
y_nn = np.hstack([np.zeros(n_reposo), np.ones(n_im)])

X_tr, X_te, y_tr, y_te = train_test_split(X_nn, y_nn, test_size=0.25,
                                            stratify=y_nn, random_state=0)
scaler = StandardScaler().fit(X_tr)
Xtr_s  = scaler.transform(X_tr)
Xte_s  = scaler.transform(X_te)

# Entrenar nuestro MLP en NumPy
mlp = MLP_NumPy(tamanos_capa=[feat_dim, 32, 16, 1], lr=0.05)
print('Entrenando MLP en NumPy  [10 → 32 → 16 → 1]:')
hist = mlp.fit(Xtr_s, y_tr, n_epochs=300, verbose=True)

p_test = mlp.predict_proba(Xte_s)
auroc  = roc_auc_score(y_te, p_test)
print(f'\nAUROC en test (MLP NumPy): {auroc:.4f}')

## Parte 3 — Verificación de gradientes

Antes de confiar en cualquier implementación de retropropagación, verifícala
contra gradientes numéricos:
$$\frac{\partial L}{\partial w} \approx \frac{L(w+\epsilon) - L(w-\epsilon)}{2\epsilon}$$

In [ ]:
def verificar_gradiente(model, X_check, y_check, eps=1e-5):
    """Compara gradientes analíticos con diferencias finitas numéricas."""
    # Calcular gradientes analíticos
    model.forward(X_check)
    gW_anal, gb_anal = model.backward(y_check)

    errores_max_rel = []
    for l in range(model.L):
        W = model.weights[l]
        # Verificar un subconjunto aleatorio de pesos
        n_check = min(20, W.size)
        indices = np.random.choice(W.size, n_check, replace=False)

        grad_num  = np.zeros(n_check)
        grad_anal = gW_anal[l].ravel()[indices]

        for idx_pos, flat_idx in enumerate(indices):
            W_flat = W.ravel()
            # +ε
            W_flat[flat_idx] += eps
            W[:] = W_flat.reshape(W.shape)
            p_mas = np.clip(model.forward(X_check), 1e-12, 1-1e-12)
            L_mas = -np.mean(y_check*np.log(p_mas) + (1-y_check)*np.log(1-p_mas))
            # -ε
            W_flat[flat_idx] -= 2*eps
            W[:] = W_flat.reshape(W.shape)
            p_menos = np.clip(model.forward(X_check), 1e-12, 1-1e-12)
            L_menos = -np.mean(y_check*np.log(p_menos) + (1-y_check)*np.log(1-p_menos))
            # Restaurar
            W_flat[flat_idx] += eps
            W[:] = W_flat.reshape(W.shape)

            grad_num[idx_pos] = (L_mas - L_menos) / (2 * eps)

        rel_err = np.abs(grad_anal - grad_num) / (np.abs(grad_anal) + np.abs(grad_num) + 1e-15)
        errores_max_rel.append(rel_err.max())
        print(f'  Capa {l+1}  error relativo máx = {rel_err.max():.2e}  '
              f'({"✅ PASA" if rel_err.max() < 1e-4 else "❌ FALLA"})')

    return errores_max_rel

np.random.seed(1)
mlp_pequeno = MLP_NumPy(tamanos_capa=[feat_dim, 8, 4, 1], lr=0.01)
X_tiny = Xtr_s[:8]
y_tiny = y_tr[:8]

print('Verificación de gradiente (analítico vs numérico):')
errs = verificar_gradiente(mlp_pequeno, X_tiny, y_tiny)
print(f'Todas las capas pasan: {all(e < 1e-4 for e in errs)}')

## Parte 4 — Optimizador Adam desde cero

Adam mantiene **tasas de aprendizaje adaptativas por parámetro** vía estimaciones
del primer y segundo momento:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t} \qquad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$
$$w_t = w_{t-1} - \frac{\eta}{\sqrt{\hat{v}_t}+\epsilon}\hat{m}_t$$

In [ ]:
class MLP_Adam(MLP_NumPy):
    """MLP con optimizador Adam."""
    def __init__(self, tamanos_capa, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, seed=0):
        super().__init__(tamanos_capa, lr=lr, seed=seed)
        self.beta1, self.beta2, self.eps_adam = beta1, beta2, eps
        self.t = 0
        # Buffers de momento
        self.mW = [np.zeros_like(w) for w in self.weights]
        self.vW = [np.zeros_like(w) for w in self.weights]
        self.mb = [np.zeros_like(b) for b in self.biases]
        self.vb = [np.zeros_like(b) for b in self.biases]

    def update(self, grads_W, grads_b):
        self.t += 1
        lr_t = self.lr * np.sqrt(1 - self.beta2**self.t) / (1 - self.beta1**self.t)
        for l in range(self.L):
            for param, grad, m, v in [
                (self.weights, grads_W, self.mW, self.vW),
                (self.biases,  grads_b, self.mb, self.vb),
            ]:
                m[l] = self.beta1 * m[l] + (1-self.beta1) * grad[l]
                v[l] = self.beta2 * v[l] + (1-self.beta2) * grad[l]**2
                param[l] -= lr_t * m[l] / (np.sqrt(v[l]) + self.eps_adam)

# Comparar optimizadores
optimizadores = {
    'SGD (lr=0.05)' : MLP_NumPy(  [feat_dim,32,16,1], lr=0.05,  seed=0),
    'SGD (lr=0.001)': MLP_NumPy(  [feat_dim,32,16,1], lr=0.001, seed=0),
    'Adam (lr=1e-3)': MLP_Adam(   [feat_dim,32,16,1], lr=1e-3,  seed=0),
    'Adam (lr=1e-2)': MLP_Adam(   [feat_dim,32,16,1], lr=1e-2,  seed=0),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores = ['steelblue','cornflowerblue','tomato','salmon']

for (nombre, modelo), color in zip(optimizadores.items(), colores):
    hist = modelo.fit(Xtr_s, y_tr, n_epochs=400, verbose=False)
    axes[0].semilogy(hist, lw=2, color=color, label=nombre)
    auroc_opt = roc_auc_score(y_te, modelo.predict_proba(Xte_s))
    axes[1].bar(nombre, auroc_opt, color=color, edgecolor='white')

axes[0].set(xlabel='Época', ylabel='Pérdida de entrenamiento (log)',
            title='Comparación de optimizadores — convergencia')
axes[0].legend(fontsize=8)
axes[1].set(ylabel='AUROC en test', title='Comparación de optimizadores — rendimiento en test',
            ylim=(0.5, 1.0))
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Parte 5 — Replicación en PyTorch y verificación con autograd

In [ ]:
if HAS_TORCH:
    import torch, torch.nn as nn, torch.optim as optim

    # Convertir datos
    Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32)
    ytr_t = torch.tensor(y_tr,  dtype=torch.float32).unsqueeze(1)
    Xte_t = torch.tensor(Xte_s, dtype=torch.float32)
    yte_t = torch.tensor(y_te,  dtype=torch.float32).unsqueeze(1)

    class MLP_PyTorch(nn.Module):
        def __init__(self, tamanos_capa):
            super().__init__()
            capas = []
            for i in range(len(tamanos_capa)-1):
                capas.append(nn.Linear(tamanos_capa[i], tamanos_capa[i+1]))
                if i < len(tamanos_capa) - 2:
                    capas.append(nn.ReLU())
            capas.append(nn.Sigmoid())
            self.net = nn.Sequential(*capas)
            # Inicialización He
            for m in self.net.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                    nn.init.zeros_(m.bias)

        def forward(self, x):
            return self.net(x)

    model_pt  = MLP_PyTorch([feat_dim, 32, 16, 1])
    optimizer = optim.Adam(model_pt.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    hist_pt = []
    for ep in range(400):
        model_pt.train()
        optimizer.zero_grad()
        p_hat = model_pt(Xtr_t)
        loss  = criterion(p_hat, ytr_t)
        loss.backward()
        optimizer.step()
        hist_pt.append(loss.item())

    model_pt.eval()
    with torch.no_grad():
        p_test_pt = model_pt(Xte_t).squeeze().numpy()
    auroc_pt = roc_auc_score(y_te, p_test_pt)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.semilogy(hist_pt, 'b-', lw=2, label=f'PyTorch Adam  (AUROC={auroc_pt:.3f})')
    ax.set(xlabel='Época', ylabel='Pérdida (log)', title='Curva de entrenamiento — MLP en PyTorch')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'AUROC en test (PyTorch MLP): {auroc_pt:.4f}')
else:
    print('PyTorch no disponible — instalar y volver a ejecutar esta celda.')

## Parte 6 — Aproximación universal: intuición

Una red con una sola capa oculta y suficientes neuronas puede aproximar cualquier
función continua en un dominio compacto.

In [ ]:
# Aproximar una función objetivo 1-D variando el ancho de la capa oculta
def funcion_objetivo(x):
    """Ejemplo biomédico: potencia alfa de EEG en función del nivel de alerta."""
    return np.sin(2*np.pi*x) * np.exp(-0.5*x**2) + 0.5*np.cos(5*x)

x_train_1d = np.linspace(-3, 3, 80).reshape(-1, 1)
y_train_1d = funcion_objetivo(x_train_1d.ravel()) + rng.normal(0, 0.15, 80)
x_test_1d  = np.linspace(-3, 3, 300).reshape(-1, 1)
y_test_1d  = funcion_objetivo(x_test_1d.ravel())

anchos = [2, 8, 32, 128]
fig, axes = plt.subplots(1, len(anchos), figsize=(16, 4), sharey=True)

if HAS_TORCH:
    from torch.utils.data import DataLoader, TensorDataset

    Xtr1 = torch.tensor(x_train_1d, dtype=torch.float32)
    ytr1 = torch.tensor(y_train_1d, dtype=torch.float32).unsqueeze(1)
    Xte1 = torch.tensor(x_test_1d,  dtype=torch.float32)

    for ax, w in zip(axes, anchos):
        net_w = nn.Sequential(
            nn.Linear(1, w), nn.Tanh(),
            nn.Linear(w, w), nn.Tanh(),
            nn.Linear(w, 1)
        )
        opt_w = optim.Adam(net_w.parameters(), lr=3e-3)
        for _ in range(2000):
            opt_w.zero_grad()
            nn.MSELoss()(net_w(Xtr1), ytr1).backward()
            opt_w.step()

        with torch.no_grad():
            y_pred_w = net_w(Xte1).squeeze().numpy()

        mse = np.mean((y_pred_w - y_test_1d)**2)
        ax.scatter(x_train_1d, y_train_1d, s=15, color='gray', alpha=0.6, zorder=2)
        ax.plot(x_test_1d, y_test_1d,  'k--', lw=2, label='f(x) verdadera')
        ax.plot(x_test_1d, y_pred_w,   'r-',  lw=2, label=f'MLP  MSE={mse:.3f}')
        ax.set(title=f'Ancho = {w}', xlabel='x')
        ax.legend(fontsize=8)

    axes[0].set_ylabel('y')
    plt.suptitle('Aproximación universal — redes más anchas ≈ menor sesgo', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('Requiere PyTorch para esta visualización.')

## ✏️ Ejercicios

1. **Retropropagación a mano.** Para una red de 2 capas con una sola neurona oculta y
   activaciones sigmoide, deriva a mano todas las derivadas parciales para un único
   ejemplo de entrenamiento. Verifica con la verificación de gradiente de tu implementación NumPy.

2. **Gradientes que explotan.** Modifica `MLP_NumPy` para usar activaciones sigmoide en
   todas las capas (en vez de ReLU). Entrena en el mismo dataset con pesos iniciales grandes.
   Grafica la norma del gradiente por capa por época. Implementa recorte de gradiente
   (gradient clipping) y demuestra que estabiliza el entrenamiento.

3. **Efecto del tamaño de batch.** Compara curvas de entrenamiento para tamaños de batch
   ∈ {1, 8, 32, 256, batch completo} en el dataset EEG. Grafica (a) pérdida vs época,
   (b) pérdida vs evaluaciones de gradiente (proxy de tiempo real), (c) AUROC final en test.
   ¿Qué tamaño de batch da la mejor generalización?

4. **Programación de tasa de aprendizaje.** Implementa el recocido coseno (cosine annealing):
   $\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max}-\eta_{min})(1 + \cos(\pi t/T_{max}))$.
   Compara con tasa de aprendizaje constante en una red de 4 capas.

5. *(Desafío)* **Construir micrograd.** Siguiendo el micrograd de Karpathy, implementa un
   motor de autograd con valores escalares — una clase `Value` que soporte `+`, `*`, `**`,
   `tanh` y sus pasos hacia atrás. Úsalo para entrenar un MLP diminuto en datos de
   clasificación 2D. Este ejercicio construye comprensión profunda de cómo funciona
   realmente el autograd de PyTorch.

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| BCI Comp IV 2a | https://www.bbci.de/competition/iv/ | Imaginería motora EEG |
| Repositorio micrograd | https://github.com/karpathy/micrograd | Para el ejercicio 5 |
| Tutoriales PyTorch | https://pytorch.org/tutorials/ | Guía oficial de inicio |